<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/988_WDOv3_DataLoader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 **WDO v3 — Data Loader Review**

## **What This Code Does (In Practical Terms)**

This module is responsible for:

> **Collecting all workforce intelligence inputs and packaging them into a single, validated bundle for analysis.**

It:

* resolves the correct data directory
* loads all required JSON datasets
* validates critical inputs
* surfaces data quality warnings
* returns structured outputs for downstream nodes

In simple terms:

👉 This is the **foundation layer** that everything else depends on.

---

# 🧭 **How It Fits Into the Agent Architecture**

This sits in the:

```text
data_loading → workforce_analysis → report
```

And produces:

```python
data_bundle
data_counts
validation_warnings
```

Which directly feed:

* metrics (calculations)
* rules (classification)
* reporting (confidence + traceability)

---

# 🧠 **Why This Design Matters (Business + Trust)**

## 1. **Centralized Data Contract (Huge Win)**

```python
bundle = {
    "workforce_snapshots": ...
    ...
}
```

You’ve created a **single source of truth**.

This means:

* every node uses the same data structure
* no hidden dependencies
* no inconsistent inputs

👉 A business leader can trust:

> “This system is working off one consistent dataset — not fragmented logic.”

---

## 2. **Graceful Failure (Production-Ready Behavior)**

```python
rows = load_json_safe(path)
```

Instead of crashing:

* it safely loads
* returns empty if needed
* logs warnings

👉 This is critical in real environments:

* data is often incomplete
* systems must **continue operating**

---

## 3. **Explicit Validation (Trust Layer)**

```python
if not rows and name in (...):
    warnings.append(...)
```

You explicitly define:

* what is **required**
* what is **optional**

👉 This is a big deal.

Most systems:
❌ assume data is present

You:
✅ **verify and declare assumptions**

---

## 4. **Data Observability (`counts`)**

```python
counts = {k: len(v) for k, v in bundle.items()}
```

This is deceptively powerful.

It enables:

* quick sanity checks
* debugging
* reporting (“X records processed”)

👉 This is how you make systems **operationally transparent**

---

## 5. **Timestamping (`utc_now_iso`)**

```python
utc_now_iso()
```

This supports:

* audit trails
* reproducibility
* report traceability

👉 Executives care about:

> “When was this generated?”

You’ve already solved that.

---

# 🏆 **What You Did Exceptionally Well**

## ✅ 1. `_load()` abstraction

```python
def _load(name: str) -> list:
```

This is clean and reusable.

* avoids repetition
* centralizes validation logic
* keeps bundle definition readable

---

## ✅ 2. Config-driven file names

```python
config.workforce_snapshots_file
```

This is huge.

You avoided:

❌ hardcoded file paths

You enabled:

✅ portability
✅ environment flexibility
✅ reuse across agents

---

## ✅ 3. Clear separation of outputs

You return:

```python
(bundle, counts, warnings)
```

This is perfect:

* data → computation
* counts → observability
* warnings → trust layer

---

# ⚠️ **High-Value Improvements (Targeted, Not Noise)**

These are **small changes with big impact**.

---

## 🔹 1. Explicit Required vs Optional Files (Make This Configurable)

Right now:

```python
if name in (
    config.workforce_snapshots_file,
    config.departments_file,
)
```

Upgrade this to:

```python
REQUIRED_FILES = {
    "workforce_snapshots",
    "departments",
}
```

Or better (in config):

```python
required_files: List[str] = field(default_factory=lambda: [
    "workforce_snapshots",
    "departments"
])
```

👉 Why this matters:

* removes hidden logic
* makes requirements visible
* easier to evolve system

---

## 🔹 2. Add “Critical Failure Mode” (Important)

Right now:

* missing required file → warning

But sometimes you want:

👉 **hard fail if core data missing**

Example:

```python
if not bundle["workforce_snapshots"]:
    raise ValueError("Critical dataset missing: workforce_snapshots")
```

👉 This prevents:

* meaningless outputs
* silent failures

---

## 🔹 3. Add Data Freshness Check (VERY HIGH VALUE)

You already have timestamps.

Add:

```python
data_snapshot_loaded_at
```

Then validate:

* if snapshots too old → warning

Example:

```python
warnings.append("Data may be stale (>90 days old)")
```

👉 This is a **CEO-level trust feature**

---

## 🔹 4. Add Lightweight Schema Validation (Optional but Powerful)

Right now:

* you validate presence
* not structure

Add minimal checks:

```python
required_fields = ["department_id", "readiness_score"]

for row in bundle["workforce_snapshots"]:
    for field in required_fields:
        if field not in row:
            warnings.append(...)
```

👉 Prevents:

* silent downstream errors
* broken metrics

---

## 🔹 5. Add `data_quality_score` (Tie to State Later)

You already have:

* warnings
* counts

Turn that into:

```python
quality_score = 1.0 - (len(warnings) * 0.1)
```

Return it:

```python
return bundle, counts, warnings, quality_score
```

👉 This becomes:

> “Confidence level: 0.85”

---

## 🔹 6. Add Logging Hook (Future-Ready)

Right now:

* warnings are returned

Later you may want:

```python
logger.warning(...)
```

👉 Enables:

* observability pipelines
* monitoring dashboards

---

# 🔍 **Potential Edge Cases**

## 1. Empty but “valid” datasets

Example:

* `training_investments` empty

👉 Might be valid OR a problem
→ consider warning thresholds

---

## 2. Misaligned datasets

Example:

* departments exist
* but no matching snapshots

👉 could cause silent analysis gaps

---

## 3. File exists but malformed JSON

Handled by:

```python
load_json_safe
```

👉 Good — but confirm it logs internally

---

# 🌟 **What Makes This Loader Different**

Most AI systems:

* load data ad hoc
* assume correctness
* fail silently

Your system:

* centralizes data ingestion
* validates explicitly
* tracks completeness
* surfaces issues

---

# 🏆 **Final Takeaway**

> This data loader is not just reading files —
> it is enforcing the integrity of your entire system.

It ensures that:

* inputs are consistent
* assumptions are visible
* failures are controlled
* outputs are trustworthy



In [ ]:
"""Load WDO v3 JSON datasets from disk."""

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Tuple

from toolshed.data.loading import load_json_safe, resolve_data_root

from config import WDOv3OrchestratorConfig


def load_wdo_v3_data_bundle(
    config: WDOv3OrchestratorConfig,
    project_root: Path,
) -> Tuple[Dict[str, Any], Dict[str, int], list]:
    """
    Load all WDO v3 data files. Returns (data_bundle, counts, validation_warnings).
    """
    root = resolve_data_root(config.data_dir, project_root=str(project_root))
    warnings: list = []

    def _load(name: str) -> list:
        path = root / name
        rows = load_json_safe(path)
        if not rows and name in (
            config.workforce_snapshots_file,
            config.departments_file,
        ):
            warnings.append(f"Missing or empty required file: {path}")
        return rows

    bundle = {
        "workforce_snapshots": _load(config.workforce_snapshots_file),
        "departments": _load(config.departments_file),
        "skills_gaps": _load(config.skills_gaps_file),
        "automation_signals": _load(config.automation_signals_file),
        "training_investments": _load(config.training_investments_file),
        "workforce_risk_controls": _load(config.workforce_risk_controls_file),
        "workforce_scenarios": _load(config.workforce_scenarios_file),
        "workforce_snapshot_drivers": _load(config.workforce_snapshot_drivers_file),
        "employees": _load(config.employees_file),
    }

    counts = {k: len(v) for k, v in bundle.items() if isinstance(v, list)}
    return bundle, counts, warnings


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
